# B5 · Modèles de diffusion : un DDPM minimal — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b5_diffusion/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/b5_diffusion/03_notebook.ipynb`, créée par `python tools/start_chapter.py B5`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B5.Q1 à B5.Q11, B5.R1 à B5.R3, B5.1 à B5.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B5.12 à B5.18 | les mathématiques de ta librairie et leurs contrôles : plannings de bruit, bruitage en forme fermée, posterior, pas DDPM et DDIM, aller-retour avec le vrai bruit (*prochaine session de génération*) | | | |
| B | B5.19 et B5.20 | jouets 2D : un MLP débruiteur conditionné par le pas, puis le flow matching (*prochaine session de génération*) | | | |
| C | B5.21 à B5.31 | le U-Net sur MNIST et Fashion-MNIST : quatre bugs, entraînement avec EMA, DDPM contre DDIM, guidance, mémorisation, distance de Fréchet, script autonome et défi (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B5")  # your own library

## Objectifs et rappel express

- Dériver la forme fermée du bruitage et le posterior d'un pas, et calculer à la main un pas DDPM et un pas DDIM.
- Implémenter en NumPy les plannings de bruit, le bruitage et les pas d'échantillonnage, et les contrôler par l'aller-retour avec le vrai bruit.
- Écrire et entraîner un petit U-Net conditionné par le pas, l'échantillonner par DDPM et DDIM, et le conditionner par la classe avec la guidance sans classifieur.
- Évaluer un générateur (distance de Fréchet, plus proches voisins) et le situer face au VAE et au GAN.

**Rappel express.** Pas de l'article $t = 1..T$, indice du code $t - 1$. $\alpha_t = 1 - \beta_t$, $\bar\alpha_t = \prod_{s \le t} \alpha_s$, $\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1 - \bar\alpha_t}\,\boldsymbol{\varepsilon}$, $\operatorname{SNR}(t) = \bar\alpha_t / (1 - \bar\alpha_t)$. Posterior : $\tilde\beta_t = \frac{1 - \bar\alpha_{t-1}}{1 - \bar\alpha_t}\beta_t$. DDPM : $\mathbf{x}_{t-1} = \frac{1}{\sqrt{\alpha_t}}\big(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar\alpha_t}}\boldsymbol{\varepsilon}_\theta\big) + \sigma_t\mathbf{z}$, sans bruit au dernier pas. DDIM ($\eta = 0$) : $\hat{\mathbf{x}}_0 = (\mathbf{x}_t - \sqrt{1 - \bar\alpha_t}\,\boldsymbol{\varepsilon}_\theta) / \sqrt{\bar\alpha_t}$, puis $\mathbf{x}_{t'} = \sqrt{\bar\alpha_{t'}}\,\hat{\mathbf{x}}_0 + \sqrt{1 - \bar\alpha_{t'}}\,\boldsymbol{\varepsilon}_\theta$. Guidance : $\tilde{\boldsymbol{\varepsilon}} = (1 + w)\,\boldsymbol{\varepsilon}_\theta(\mathbf{x}_t, c) - w\,\boldsymbol{\varepsilon}_\theta(\mathbf{x}_t, \varnothing)$, `guidance_scale` $= 1 + w$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B5.1 à B5.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; un vecteur : une liste de nombres (`[0.5, -1.25]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ B5.6 et B5.7, l'estimation 🧮 B5.8, la réflexion (🗣️ B5.9, ⚖️ B5.10, 📄 B5.11) et l'entretien se corrigent avec `05_solutions.md`.

**B5.Q1 — Diffusion, VAE, GAN : comment chacun fabrique-t-il une image ?**

In [ ]:
# B5.Q1: the values you found on paper
answer_B5_Q1a = ...  # a) the letter of your choice, e.g. "E"
answer_B5_Q1b = ...  # b) True or False
answer_B5_Q1c = ...  # c) the letter of your choice, e.g. "E"
answer_B5_Q1d = ...  # d) True or False
answer_B5_Q1e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B5_Q1a, answer_B5_Q1b, answer_B5_Q1c, answer_B5_Q1d, answer_B5_Q1e]):
    wb.check(f"B5.Q1{letter}", answer)

**B5.Q2 — Pourquoi détruire l'image en mille petits pas de bruit plutôt qu'en un seul ?**

In [ ]:
# B5.Q2: the values you found on paper
answer_B5_Q2a = ...  # a) the letter of your choice, e.g. "E"
answer_B5_Q2b = ...  # b) a number, 2 decimals
answer_B5_Q2c = ...  # c) a number, 2 decimals
answer_B5_Q2d = ...  # d) True or False
answer_B5_Q2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B5_Q2a, answer_B5_Q2b, answer_B5_Q2c, answer_B5_Q2d, answer_B5_Q2e]):
    wb.check(f"B5.Q2{letter}", answer)

**B5.Q3 — ᾱ_t : quelle part de l'image reste-t-il au pas t ?**

In [ ]:
# B5.Q3: the values you found on paper
answer_B5_Q3a = ...  # a) a number, 2 decimals
answer_B5_Q3b = ...  # b) a number, 2 decimals
answer_B5_Q3c = ...  # c) True or False
answer_B5_Q3d = ...  # d) a number, 2 decimals
answer_B5_Q3e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B5_Q3a, answer_B5_Q3b, answer_B5_Q3c, answer_B5_Q3d, answer_B5_Q3e]):
    wb.check(f"B5.Q3{letter}", answer)

**B5.Q4 — Prédire le bruit, l'image ou la vitesse : la même information ?**

In [ ]:
# B5.Q4: the values you found on paper
answer_B5_Q4a = ...  # a) the letter of your choice, e.g. "E"
answer_B5_Q4b = ...  # b) a number, 3 decimals
answer_B5_Q4c = ...  # c) True or False
answer_B5_Q4d = ...  # d) the letter of your choice, e.g. "E"
answer_B5_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B5_Q4a, answer_B5_Q4b, answer_B5_Q4c, answer_B5_Q4d, answer_B5_Q4e]):
    wb.check(f"B5.Q4{letter}", answer)

**B5.Q5 — Le pas de temps est une entrée du réseau : pourquoi, et comment l'encoder ?**

In [ ]:
# B5.Q5: the values you found on paper
answer_B5_Q5a = ...  # a) the letter of your choice, e.g. "E"
answer_B5_Q5b = ...  # b) True or False
answer_B5_Q5c = ...  # c) a whole number
answer_B5_Q5d = ...  # d) the letter of your choice, e.g. "E"
answer_B5_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B5_Q5a, answer_B5_Q5b, answer_B5_Q5c, answer_B5_Q5d, answer_B5_Q5e]):
    wb.check(f"B5.Q5{letter}", answer)

**B5.Q6 — U-Net : à quoi servent les connexions de saut ?**

In [ ]:
# B5.Q6: the values you found on paper
answer_B5_Q6a = ...  # a) the letter of your choice, e.g. "E"
answer_B5_Q6b = ...  # b) a whole number
answer_B5_Q6c = ...  # c) True or False
answer_B5_Q6d = ...  # d) a whole number
answer_B5_Q6e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B5_Q6a, answer_B5_Q6b, answer_B5_Q6c, answer_B5_Q6d, answer_B5_Q6e]):
    wb.check(f"B5.Q6{letter}", answer)

**B5.Q7 — Échantillonnage DDPM : pourquoi est-il lent et aléatoire ?**

In [ ]:
# B5.Q7: the values you found on paper
answer_B5_Q7a = ...  # a) a whole number
answer_B5_Q7b = ...  # b) True or False
answer_B5_Q7c = ...  # c) True or False
answer_B5_Q7d = ...  # d) the letter of your choice, e.g. "E"
answer_B5_Q7e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B5_Q7a, answer_B5_Q7b, answer_B5_Q7c, answer_B5_Q7d, answer_B5_Q7e]):
    wb.check(f"B5.Q7{letter}", answer)

**B5.Q8 — DDIM : moins de pas avec le même réseau, vrai ou faux**

In [ ]:
# B5.Q8: the values you found on paper
answer_B5_Q8a = ...  # a) True or False
answer_B5_Q8b = ...  # b) True or False
answer_B5_Q8c = ...  # c) True or False
answer_B5_Q8d = ...  # d) True or False
answer_B5_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B5_Q8a, answer_B5_Q8b, answer_B5_Q8c, answer_B5_Q8d, answer_B5_Q8e]):
    wb.check(f"B5.Q8{letter}", answer)

**B5.Q9 — Guidance sans classifieur : ce que règle l'échelle w**

In [ ]:
# B5.Q9: the values you found on paper
answer_B5_Q9a = ...  # a) a number, 1 decimal
answer_B5_Q9b = ...  # b) the letter of your choice, e.g. "E"
answer_B5_Q9c = ...  # c) True or False
answer_B5_Q9d = ...  # d) a whole number
answer_B5_Q9e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B5_Q9a, answer_B5_Q9b, answer_B5_Q9c, answer_B5_Q9d, answer_B5_Q9e]):
    wb.check(f"B5.Q9{letter}", answer)

**B5.Q10 — Un bon FID peut-il cacher de la copie des données d'entraînement ?**

In [ ]:
# B5.Q10: the values you found on paper
answer_B5_Q10a = ...  # a) True or False
answer_B5_Q10b = ...  # b) the letter of your choice, e.g. "E"
answer_B5_Q10c = ...  # c) True or False
answer_B5_Q10d = ...  # d) the letter of your choice, e.g. "E"
answer_B5_Q10e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B5_Q10a, answer_B5_Q10b, answer_B5_Q10c, answer_B5_Q10d, answer_B5_Q10e]):
    wb.check(f"B5.Q10{letter}", answer)

**B5.Q11 — Diffusion latente, transformers de diffusion, flow matching : ce qui a changé depuis 2020**

In [ ]:
# B5.Q11: the values you found on paper
answer_B5_Q11a = ...  # a) a whole number
answer_B5_Q11b = ...  # b) the letter of your choice, e.g. "E"
answer_B5_Q11c = ...  # c) the letter of your choice, e.g. "E"
answer_B5_Q11d = ...  # d) True or False
answer_B5_Q11e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B5_Q11a, answer_B5_Q11b, answer_B5_Q11c, answer_B5_Q11d, answer_B5_Q11e]):
    wb.check(f"B5.Q11{letter}", answer)

**B5.R1 — L'encodage sinusoïdal des positions**

In [ ]:
# B5.R1: the values you found on paper
answer_B5_R1a = ...  # a) a whole number
answer_B5_R1b = ...  # b) a number, 2 decimals
answer_B5_R1c = ...  # c) True or False
answer_B5_R1d = ...  # d) True or False
answer_B5_R1e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B5_R1a, answer_B5_R1b, answer_B5_R1c, answer_B5_R1d, answer_B5_R1e]):
    wb.check(f"B5.R1{letter}", answer)

**B5.R2 — Ce que mesure la distance de Fréchet et l'effondrement de mode**

In [ ]:
# B5.R2: the values you found on paper
answer_B5_R2a = ...  # a) a whole number
answer_B5_R2b = ...  # b) the letter of your choice, e.g. "E"
answer_B5_R2c = ...  # c) True or False
answer_B5_R2d = ...  # d) True or False
answer_B5_R2e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B5_R2a, answer_B5_R2b, answer_B5_R2c, answer_B5_R2d, answer_B5_R2e]):
    wb.check(f"B5.R2{letter}", answer)

**B5.R3 — L'astuce de reparamétrisation et le terme KL du VAE**

In [ ]:
# B5.R3: the values you found on paper
answer_B5_R3a = ...  # a) a number, 2 decimals
answer_B5_R3b = ...  # b) a number, 4 decimals
answer_B5_R3c = ...  # c) the letter of your choice, e.g. "E"
answer_B5_R3d = ...  # d) True or False
answer_B5_R3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B5_R3a, answer_B5_R3b, answer_B5_R3c, answer_B5_R3d, answer_B5_R3e]):
    wb.check(f"B5.R3{letter}", answer)

**Ex B5.1 — Guidance sans classifieur : combiner les deux prédictions pour w = 0, 1 et 3**

In [ ]:
# Ex B5.1: the values you found on paper
answer_B5_1a = ...  # a) a list of two numbers, 2 decimals
answer_B5_1b = ...  # b) a list of two numbers, 2 decimals
answer_B5_1c = ...  # c) a list of two numbers, 2 decimals
answer_B5_1d = ...  # d) a whole number
answer_B5_1e = ...  # e) a number, 2 decimals
answer_B5_1f = ...  # f) True or False
answer_B5_1g = ...  # g) True or False
answer_B5_1h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_B5_1a, answer_B5_1b, answer_B5_1c, answer_B5_1d, answer_B5_1e, answer_B5_1f, answer_B5_1g, answer_B5_1h]):
    wb.check(f"B5.1{letter}", answer)

**Ex B5.2 — Planning linéaire : β_t, α_t et ᾱ_t pour T = 4, puis ᾱ_T pour T = 1 000**

In [ ]:
# Ex B5.2: the values you found on paper
answer_B5_2a = ...  # a) a number, 2 decimals
answer_B5_2b = ...  # b) a number, 4 decimals
answer_B5_2c = ...  # c) a number, 4 decimals
answer_B5_2d = ...  # d) a number, 4 decimals
answer_B5_2e = ...  # e) a number, 4 decimals
answer_B5_2f = ...  # f) True or False
answer_B5_2g = ...  # g) a number, 2 decimals
answer_B5_2h = ...  # h) a number, 2 decimals
answer_B5_2i = ...  # i) a number, 2 decimals
answer_B5_2j = ...  # j) a number, 2 decimals
answer_B5_2k = ...  # k) a number, 2 decimals
answer_B5_2l = ...  # l) True or False

for letter, answer in zip("abcdefghijkl", [answer_B5_2a, answer_B5_2b, answer_B5_2c, answer_B5_2d, answer_B5_2e, answer_B5_2f, answer_B5_2g, answer_B5_2h, answer_B5_2i, answer_B5_2j, answer_B5_2k, answer_B5_2l]):
    wb.check(f"B5.2{letter}", answer)

**Ex B5.3 — Rapport signal sur bruit : à quel pas un chiffre devient-il illisible, planning linéaire ou cosinus ?**

In [ ]:
# Ex B5.3: the values you found on paper
answer_B5_3a = ...  # a) a number, 3 decimals
answer_B5_3b = ...  # b) a number, 2 decimals
answer_B5_3c = ...  # c) a number, 2 decimals
answer_B5_3d = ...  # d) a number, 2 decimals
answer_B5_3e = ...  # e) a number, 2 decimals
answer_B5_3f = ...  # f) a number, 2 decimals
answer_B5_3g = ...  # g) a number, 2 decimals
answer_B5_3h = ...  # h) True or False
answer_B5_3i = ...  # i) True or False
answer_B5_3j = ...  # j) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdefghij", [answer_B5_3a, answer_B5_3b, answer_B5_3c, answer_B5_3d, answer_B5_3e, answer_B5_3f, answer_B5_3g, answer_B5_3h, answer_B5_3i, answer_B5_3j]):
    wb.check(f"B5.3{letter}", answer)

**Ex B5.4 — Un pas DDPM à la main, en dimension 1, avec un bruit prédit donné**

In [ ]:
# Ex B5.4: the values you found on paper
answer_B5_4a = ...  # a) a number, 4 decimals
answer_B5_4b = ...  # b) a number, 4 decimals
answer_B5_4c = ...  # c) a number, 4 decimals
answer_B5_4d = ...  # d) a number, 4 decimals
answer_B5_4e = ...  # e) a number, 4 decimals
answer_B5_4f = ...  # f) a number, 4 decimals
answer_B5_4g = ...  # g) a number, 4 decimals
answer_B5_4h = ...  # h) True or False
answer_B5_4i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_B5_4a, answer_B5_4b, answer_B5_4c, answer_B5_4d, answer_B5_4e, answer_B5_4f, answer_B5_4g, answer_B5_4h, answer_B5_4i]):
    wb.check(f"B5.4{letter}", answer)

**Ex B5.5 — Un pas DDIM à la main et l'image estimée x̂_0**

In [ ]:
# Ex B5.5: the values you found on paper
answer_B5_5a = ...  # a) a number, 4 decimals
answer_B5_5b = ...  # b) a number, 4 decimals
answer_B5_5c = ...  # c) the letter of your choice, e.g. "E"
answer_B5_5d = ...  # d) a number, 4 decimals
answer_B5_5e = ...  # e) True or False
answer_B5_5f = ...  # f) a number, 4 decimals
answer_B5_5g = ...  # g) a number, 4 decimals
answer_B5_5h = ...  # h) a number, 4 decimals
answer_B5_5i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_B5_5a, answer_B5_5b, answer_B5_5c, answer_B5_5d, answer_B5_5e, answer_B5_5f, answer_B5_5g, answer_B5_5h, answer_B5_5i]):
    wb.check(f"B5.5{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer à la main un planning de bruit, ses $\bar\alpha_t$ et son rapport signal sur bruit ?
2. Sais-tu dériver la forme fermée et le posterior d'un pas, puis calculer un pas DDPM et un pas DDIM ?
3. Sais-tu appliquer la guidance sans classifieur et traduire `guidance_scale` en $w$ ?
4. Sais-tu estimer le coût d'un échantillonnage, et ce que gagnent DDIM, la distillation et l'espace latent ?

**Pour aller plus loin** : l'article de Ho, Jain et Abbeel et le billet de L. Weng, cités dans la fiche. La suite : le bonus B6, l'explicabilité et l'équité.

*Les exercices de code B5.12 à B5.31 (mathématiques de ta librairie, jouets 2D, U-Net sur MNIST et Fashion-MNIST) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B5` pour obtenir le notebook complet.*